# Problem A — the result: the table wins on simplicity, not capability

Problem A is a small, exactly *tabulatable* decision. The comparison here is between two
**reinforcement learners** scored against the analytic basic-strategy table (100% by definition): a
**tabular Q-learner** (lookup-table representation) and a **DQN** (neural representation). Both are
trained and scored on the **same action set — hit / stand / double only** (no split, no surrender in
either), on the same noisy ±2 rewards. Only the representation differs. All numbers below are **means
over runs** (with ranges), not a best-of cherry-pick, and we report **two metrics**: agreement (cell
match) and edge (money lost per hand — the one that actually costs).

1. The **tabular Q-learner** (no-split, 5M episodes) averages **~0.93 agreement / ~1.3% edge** —
   strong, though even it does not perfectly reproduce basic strategy and it scatters run to run.
2. A **naive** DQN plateaus at **~0.82 / ~1.9% edge**, with a wide run-to-run band (~7 agreement pts;
   even fixed seed + config reruns scatter), and the usual knobs move it less than that band.
3. A **stacked** DQN — a decaying learning rate (the one clean lever) plus a stack of supporting
   choices (scalar encoding, Double-DQN, larger batch, longer training; and a dealer control variate
   that turns out inert, 04) — reaches **~0.89 agreement / ~1.15% edge**. On agreement it closes ~65%
   of the naive→tabular gap and lands ~4 pts short; **on edge it is statistically tied with the
   tabular learner** (its mean is even a touch lower).

Both RL agents sit well above the **measured ~0.1%/hand basic-strategy floor** — the in-harness
fresh-shoe optimum (the literature puts a brand-new shoe near ~0.02%, and at 200k hands the two are
indistinguishable). Learning the map from noisy ±2 rewards costs ~1%/hand vs simply *knowing* the
table — but the two learners sit there *together*. Most of the naive deficit is *instability and
rare-cell starvation* (02–04), which the stack recovers. The **residual ~4 agreement points is a
genuine representation floor** — a smooth approximator can't reproduce the table's exact decision
boundaries (05–06) — but a *cheap* one: it lives in razor-thin boundary cells, which is exactly why on
the money metric there is essentially no gap between them. The honest verdict: **the net matches the
tabular learner on edge and nearly on agreement, but only with a decaying-step fix plus a stack of
choices the table never needs, and it still wobbles seed to seed and cannot draw the boundaries
exactly. The table representation wins on agreement-exactness, robustness, and zero tuning — not on the
money capability.** Notebooks 02–06 show *why* it plateaus, *what* recovers it, what the residual is,
and that it comes down to a representation choice (smooth vs sharp).

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from blackjack_rl.analysis_loader import load_runs

df = load_runs(); dqn = df[df.method == 'dqn']
# anchor = tabular Q-LEARNER, NO-SPLIT (with_splits=False) to match the DQN's hit/stand/double action
# set, well-trained (5M episodes). Split-enabled tabular runs exist (added later) but solve a larger
# problem (~52 split decisions, 364 cells) and post a better edge because split is +EV; including them
# would be an unfair anchor, so they are excluded. Mean over runs, not best-of.
tab = df[(df.method == 'tabular') & (df.episodes == 5_000_000) & (~df.with_splits)]
# optimal basic-strategy house edge, MEASURED in the same fresh-shoe 200k harness as the agents (paired
# by seed inside each run) — not a hardcoded constant. The per-run standard error is ~0.3%, so this
# number is statistically at the literature fresh-shoe floor (~0.02%); it reads ~0.1% here. Basic plays
# the FULL action set (it may split); the no-split agents cannot, so part of their edge gap is simply
# actions they were never given.
basic_pool = df[(~df.with_splits) & df.basic_edge_pct.notna()]
BASIC_EDGE = basic_pool.basic_edge_pct.mean()

naive = dqn[(dqn.hidden=='[64, 64]') & (dqn.episodes==1_000_000) & (~dqn.swa)
            & (dqn.lr_schedule=='constant') & (dqn.reward_baseline=='none') & (dqn.double_after==0)]
naive = naive[naive.edge_pct < 4.0]  # drop one divergent run (6.2% edge, ~2x the next) per review
stacked = dqn[(dqn.encoding=='scalar') & (dqn.lr_schedule=='linear') & dqn.double_dqn
              & (dqn.reward_baseline=='stand') & (dqn.episodes==1_500_000) & (dqn.batch==512)]
assert len(naive) and len(stacked) and len(tab), 'run selection empty — check field formats'
TABLE = tab.agreement.mean()
for name, g in [('tabular Q-learner', tab), ('naive DQN', naive), ('stacked DQN', stacked)]:
    print(f'{name:18} n={len(g):2}  agree {g.agreement.mean():.1%} ({g.agreement.min():.0%}-{g.agreement.max():.0%})'
          f'   edge {g.edge_pct.mean():.2f}% ({g.edge_pct.min():.2f}-{g.edge_pct.max():.2f})')
print(f'basic strategy floor (measured, fresh-shoe): {BASIC_EDGE:.2f}%/hand  (+-~0.3% per-run SE)')
gap = (stacked.agreement.mean() - naive.agreement.mean()) / (TABLE - naive.agreement.mean())
print(f'\nstack closes {gap:.0%} of the naive->tabular agreement gap; edge gap to tabular: '
      f'{stacked.edge_pct.mean()-tab.edge_pct.mean():+.2f} pts (within run scatter).')

## Two metrics, two stories

Left: **agreement** — the tabular learner leads, the stack closes most of the naive gap, ~4 points
short. Right: **edge** (lower = less money lost; dashed line = the measured basic-strategy floor,
~0.1%/hand) — the stacked DQN and the tabular learner overlap (the DQN's mean is even slightly lower),
both well above that floor, while the naive DQN clearly loses more. The table's advantage over the net
lives in agreement, not money. (Whiskers = run range; the tabular learner is not a deterministic
oracle — it scatters too.)

In [ ]:
groups = [('naive\nDQN', naive), ('stacked\nDQN', stacked), ('tabular\nQ-learner', tab)]
labels = [g[0] for g in groups]
colors = ['#888888', '#378ADD', '#1D9E75']
fig, (axA, axE) = plt.subplots(1, 2, figsize=(11, 4))
# agreement
mA = [g[1].agreement.mean() for g in groups]
loA = [g[1].agreement.mean()-g[1].agreement.min() for g in groups]; hiA = [g[1].agreement.max()-g[1].agreement.mean() for g in groups]
bA = axA.bar(labels, mA, color=colors, width=0.6)
axA.errorbar(range(3), mA, yerr=[loA, hiA], fmt='none', ecolor='#1b3a5b', capsize=6, lw=1.5)
for b, m in zip(bA, mA): axA.text(b.get_x()+b.get_width()/2, m+0.006, f'{m:.0%}', ha='center', fontsize=11)
axA.set_ylim(0.75, 0.99); axA.set_ylabel('agreement with basic strategy'); axA.set_title('Agreement (higher = better)')
# edge
mE = [g[1].edge_pct.mean() for g in groups]
loE = [g[1].edge_pct.mean()-g[1].edge_pct.min() for g in groups]; hiE = [g[1].edge_pct.max()-g[1].edge_pct.mean() for g in groups]
bE = axE.bar(labels, mE, color=colors, width=0.6)
axE.errorbar(range(3), mE, yerr=[loE, hiE], fmt='none', ecolor='#1b3a5b', capsize=6, lw=1.5)
for b, m in zip(bE, mE): axE.text(b.get_x()+b.get_width()/2, m+0.06, f'{m:.2f}%', ha='center', fontsize=11)
axE.axhline(BASIC_EDGE, ls='--', color='#1D9E75', lw=1, label=f'basic strategy ~{BASIC_EDGE:.1f}%/hand (measured floor)')
axE.set_ylabel('house edge: % lost per hand'); axE.set_title('Edge (lower = better)'); axE.legend(fontsize=8)
plt.tight_layout(); plt.show()

## It's the stack, not any single knob

Crucially, *no single lever* closes the agreement gap — on the naive baseline, encoding, Double-DQN,
and capacity all move agreement less than the run-to-run band (note encoding: one-hot is marginally
*ahead* of scalar on raw agreement here — the encodings trade rare-cell generalization for boundary
sharpness, notebook 06, not this single number). The recovery comes from **combining** the
stabilizers, led by the decaying lr (notebook 03). This is why the first-pass verdict looked like a
wall: each knob tried alone does nothing visible above the noise.

In [ ]:
band = naive.agreement.max() - naive.agreement.min()
one_m = dqn[(dqn.episodes==1_000_000) & (~dqn.swa) & (dqn.reward_baseline=='none')
            & (dqn.double_after==0) & (dqn.lr_schedule=='constant')]
print('encoding (64x64, 1M):', one_m[one_m.hidden=='[64, 64]'].groupby('encoding').agreement.mean().round(3).to_dict())
print('Double-DQN (onehot)  :', one_m[(one_m.encoding=='onehot') & (one_m.hidden=='[64, 64]')]
      .groupby('double_dqn').agreement.mean().round(3).to_dict())
print('capacity (scalar)    :', one_m[one_m.encoding=='scalar'].groupby('hidden').agreement.mean().round(3).to_dict())
print(f'\nrun-to-run band ~{band*100:.0f} pts — every single-knob difference above is inside it.')

## Notes on metrics and fairness

- **Two metrics, not one.** Agreement and edge tell different stories: the tabular learner leads on
  agreement but ties the stacked DQN on edge. Edge is the decision-relevant truth — a disagreement in
  a once-in-a-thousand cell costs almost nothing — so the agreement lead overstates the table's
  practical advantage. A *third* signal, the agent's self-reported EV-gap on disagreements, is
  misleading (smaller self-gaps did not mean a better edge); notebook 04 unpacks why.
- **The edge floor is measured here, the same way as the agents.** Optimal basic strategy is evaluated
  in *every* run on the identical fresh shoes (paired by seed, 200k hands) and averages **~0.1%/hand**
  — statistically at the literature fresh-shoe floor (~0.02%) given the ~0.3% per-run standard error.
  (Phase 2's ~0.45–0.5%/hand figure is a *different regime* — continuous, penetrated-shoe session play
  — and is not comparable to these single-hand fresh-shoe evals; the agents are measured fresh-shoe, so
  the fresh-shoe basic is the consistent floor.) Basic plays the full action set; the no-split agents
  do not, which is part of why they lose more.
- **Same action set.** Both agents are hit/stand/double only — no split or surrender in either agent's
  training or scored cells. The tabular anchor is restricted to `with_splits=False` runs; split-
  enabled tabular runs exist (added later) and post a better edge because splitting is +EV, but they
  solve a larger problem and would be an unfair anchor, so they are excluded.
- **Cell-grid caveat.** The tabular diff enumerates 280 cells vs the DQN's 240; the 40 extras are
  trivial (hard-4, soft-12, busted-21) and ~97% agree, inflating tabular agreement by only ~0.4 pts
  (best tabular run: 0.950 full grid → 0.946 on the common grid). Negligible for the verdict.
- **Outlier.** One naive run (6.2% edge, ~2× the next) is excluded from the naive aggregates.

## Conclusion

For an exactly tabulatable problem, a naive neural learner trails a no-split tabular learner on the
identical task. Most of that gap is instability plus rare-cell starvation (notebooks 02–04), which a
decaying-lr lever plus a stack of supporting choices recovers — to ~0.89 agreement and a house edge
**statistically tied with the table** (~1.1–1.3%, both well above the measured ~0.1% fresh-shoe floor).
The residual ~4 agreement points is a genuine **representation floor**: the smooth net cannot reproduce
the table's exact decision boundaries (notebooks 05–06) — but a *cheap* floor, concentrated in
razor-thin boundary cells, which is why the edge ties. And it is a representation *choice*, not a wall:
a sharper (one-hot) encoding halves those boundary errors and, on edge, can match or beat the table
(06). The table still wins — because it reaches its level *exactly, stably, and with zero tuning*,
while the net needs the stack, wobbles seed to seed, and blurs the boundaries. (Neither RL agent
perfectly reproduces basic strategy; the analytic table is the 100% / ~0.1%-edge fresh-shoe reference
both are measured against.) Right tool, for reasons of engineering cost, robustness, and exactness
rather than money capability.